In [3]:
from PIL import Image 
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

from pathlib import Path
import re

In [4]:
# Все, что связано с расчетном самого спектра
# и соответствующих срезов (по λ или по ϴ)

def gaussian(w, w0, sig):
    W = w - w0
    return np.exp(-0.5 * W**2 / sig**2)


def wl_convert(func, w_range, *params):
    """
    w_range [rad/ps]
    """
    l_range = 2 * np.pi * c / w_range * 1e-3  # [nm]
    
    y_w = func(w_range, *params)
    y_l = y_w / l_range**2
    y_l /= np.max(y_l)

    return y_l


def l_distr(x, alpha, beta, L, K):
    l = L + K * x  # связь пикселя с длиной волны в [nm]

    C = 2 * np.pi * c * 1e-3
    w = C / l  # [rad/ps]

    l0 = 810
    w0 = C / l0  # [rad/ps]

    l_fwhm = 10
    w_fwhm = C * l_fwhm / (l0**2 - (l_fwhm/2)**2)
    sig_w = w_fwhm / 2 / np.sqrt(2 * np.log(2))
    
    y_l = wl_convert(gaussian, w, w0, sig_w)
    y_l /= np.max(y_l)
    y_l = beta + alpha * y_l
    
    return y_l


# ============================================================
# 2. Sellmeier: BBO и кальцит
# ============================================================
def n_air(lambda_um):
    """
    lambda_um — vacuum wavelength in microns.
    """
    lam2 = np.asarray(lambda_um) ** 2
    B1 = 0.05792105
    B2 = 0.00167917
    C1 = 238.0185
    C2 = 57.362

    UV_1 = B1 / (C1 - 1/lambda_um**2)
    UV_2 = B2 / (C2 - 1/lambda_um**2)

    n = 1 + UV_1 + UV_2
    return n
    


def n_bbo_o(lambda_um):
    """
    Ordinary refractive index of BBO.
    lambda_um — vacuum wavelength in microns.
    """
    lam2 = np.asarray(lambda_um) ** 2
    n2 = (
        0.90291 * lam2 / (lam2 - 0.003926)
        + 0.83155 * lam2 / (lam2 - 0.018786)
        + 0.76536 * lam2 / (lam2 - 60.01)
        + 1
    )
    return np.sqrt(n2)


def n_bbo_e(lambda_um):
    """
    Extraordinary refractive index of BBO.
    lambda_um — vacuum wavelength in microns.
    """
    lam2 = np.asarray(lambda_um) ** 2
    n2 = (
        1.151075 * lam2 / (lam2 - 0.007142)
        + 0.21803 * lam2 / (lam2 - 0.02259)
        + 0.656 * lam2 / (lam2 - 263)
        + 1
    )
    return np.sqrt(n2)


def n_bbo_eff(n_o, n_e, psi_rad):
    """
    Effective refractive index for a uniaxial crystal.
    psi_rad — angle between the wave vector and the optical axis.
    """
    return 1.0 / np.sqrt(
        (np.cos(psi_rad) ** 2) / (n_o ** 2)
        +
        (np.sin(psi_rad) ** 2) / (n_e ** 2)
    )


def n_caco3_o(lambda_um):
    """
    Calcite ordinary refractive index.
    """
    lam2 = np.asarray(lambda_um) ** 2
    n2 = (
        0.73358749
        + 0.96464345 * lam2 / (lam2 - 1.94325203e-2)
        + 1.82831454 * lam2 / (lam2 - 120.0)
        + 1
    )
    return np.sqrt(n2)


def n_caco3_e(lambda_um):
    """
    Calcite extraordinary refractive index
    """
    lam2 = np.asarray(lambda_um) ** 2
    n2 = (
        0.35859695
        + 0.82427830 * lam2 / (lam2 - 1.06689543e-2)
        + 0.14429128 * lam2 / (lam2 - 120.0)
        + 1
    )
    return np.sqrt(n2)


def n_caco3(lambda_um, pol):
    if pol == "o":
        return n_caco3_o(lambda_um)
    if pol == "e":
        return n_caco3_e(lambda_um)
    raise ValueError("pol must be 'o' or 'e'")


# ============================================================
# 3. Вспомогательные функции
# ============================================================

def omega_to_lambda_um(omega):
    """
    Angular frequency omega -> vacuum wavelength in microns
    """
    return 2 * np.pi * c / omega * 1e6


def k_from_n_omega(n, omega):
    """
    Wave number in medium.
    """
    return n * omega / c


def kz_from_k_q(k, q):
    """
    Longitudinal component:
        k_z = sqrt(k^2 - q^2)
    """
    kz = np.sqrt(k**2 - q**2)
    return kz


def sinc(x):
    """
    sinc(x) = sin(x) / x.
    """
    return np.sinc(x / np.pi)


def find_bbo_cut_angle_for_degenerate_collinear():
    """
    Finds BBO cut angle for collinear degenerate type-I (e=o+o) phase matching:
        404 nm -> 808 nm + 808 nm.
    Condition:
        n_eff(lambda_p_um, psi) = n_o(2 * lambda_p_um)
    """
    lambda_p_um = 404.0 * 1e-3
    lambda_s_um = 2 * lambda_p_um

    n_s = n_bbo_o(lambda_s_um)

    n_po = n_bbo_o(lambda_p_um)
    n_pe = n_bbo_e(lambda_p_um)

    lo = 0.0
    hi = 0.5 * np.pi

    for _ in range(100):
        mid = 0.5 * (lo + hi)
        n_p_eff = n_bbo_eff(n_po, n_pe, mid)

        # n_bbo_eff goes from n_o to n_e as theta increases.
        if n_p_eff > n_s:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

# print(f"BBO psi_cut = {psi_cut_deg:.4f} deg")


# ============================================================
# 4. Фазовые синхронизмы
# ============================================================

def delta_k_bbo(omega_s, theta_s):
    """
    BBO phase mismatch for plane-wave pump.

    Independent variables:
        omega_s, theta_s.

    Then:
        omega_i = w_p - omega_s,
        q_i = -q_s.

    theta_s is the internal signal angle in BBO.
    """
    omega_i = w_p - omega_s

    lambda_p_um = omega_to_lambda_um(w_p)
    lambda_s_um = omega_to_lambda_um(omega_s)
    lambda_i_um = omega_to_lambda_um(omega_i)

    # Pump extraordinary in BBO, propagating along z.
    n_po = n_bbo_o(lambda_p_um)
    n_pe = n_bbo_e(lambda_p_um)
    n_p = n_bbo_eff(n_po, n_pe, psi_cut)

    n_s = n_bbo_o(lambda_s_um)
    n_i = n_bbo_o(lambda_i_um)
    
    # Signal/idler ordinary in BBO.
    k_p = k_from_n_omega(n_p, w_p)
    k_s = k_from_n_omega(n_s, omega_s)
    k_i = k_from_n_omega(n_i, omega_i)

    # Plane-wave pump: q_p = 0, hence q_s + q_i = 0.
    q_s = k_s * np.sin(theta_s)
    q_i = -q_s

    k_sz = kz_from_k_q(k_s, q_s)
    k_iz = kz_from_k_q(k_i, q_i)

    delta_k = k_p - k_sz - k_iz

    return delta_k, q_s


def delta_k_air(omega_s, q_s):
    """
    Longitudinal phase mismatch in air.

    Transverse wave-vectors q_s and q_i are conserved due to the Snellius law.
    They are inherited from wave synchronism condition in BBO.
    """
    omega_i = w_p - omega_s

    lambda_p_um = omega_to_lambda_um(w_p)
    lambda_s_um = omega_to_lambda_um(omega_s)
    lambda_i_um = omega_to_lambda_um(omega_i)

    n_p = n_air(lambda_p_um)
    n_s = n_air(lambda_s_um)
    n_i = n_air(lambda_i_um)
    
    k_p = k_from_n_omega(n_p, w_p)
    k_s = k_from_n_omega(n_s, omega_s)
    k_i = k_from_n_omega(n_i, omega_i)
    
    q_i = -q_s
    k_sz = kz_from_k_q(k_s, q_s)
    k_iz = kz_from_k_q(k_i, q_i)

    delta_k = k_p - k_sz - k_iz

    return delta_k


def delta_k_caco3(omega_s, q_s, pol):
    """
    Longitudinal phase mismatch in calcite.

    Default geometry:
        pump  : o-ray
        signal: e-ray
        idler : e-ray

    The transverse components q_s and q_i are inherited from
    transverse momentum conservation.
    """
    omega_i = w_p - omega_s

    lambda_p_um = omega_to_lambda_um(w_p)
    lambda_s_um = omega_to_lambda_um(omega_s)
    lambda_i_um = omega_to_lambda_um(omega_i)

    pol_p = pol["p"]
    pol_s = pol["s"]
    pol_i = pol["i"]

    n_p = n_caco3(lambda_p_um, pol_p)
    n_s = n_caco3(lambda_s_um, pol_s)
    n_i = n_caco3(lambda_i_um, pol_i)
    
    k_p = k_from_n_omega(n_p, w_p)
    k_s = k_from_n_omega(n_s, omega_s)
    k_i = k_from_n_omega(n_i, omega_i)

    q_i = -q_s
    k_sz = kz_from_k_q(k_s, q_s)
    k_iz = kz_from_k_q(k_i, q_i)
    
    delta_k = k_p - k_sz - k_iz

    return delta_k


# ============================================================
# 5. Амплитуда бифотона для плосковолновой накачки
# ============================================================

def JSA(omega_s, theta_s, L1, L2, l_air, l_caco3, calcite_pol):
    """
    Computes the biphoton amplitude:

        F = F1 + F2.

    Here:
        omega_i = w_p - omega_s,
        q_i = -q_s.

    No integration over theta_i is performed.
    """
    omega_i = w_p - omega_s

    dk_bbo, q_s = delta_k_bbo(
        omega_s=omega_s,
        theta_s=theta_s,
    )

    dk_air = delta_k_air(
        omega_s=omega_s,
        q_s=q_s
    )

    dk_caco3 = delta_k_caco3(
        omega_s=omega_s,
        q_s=q_s,
        pol=calcite_pol
    )

    # Idler angle inside BBO, found from transverse momentum conservation.
    lambda_i_um = omega_to_lambda_um(omega_i)
    k_i_bbo = k_from_n_omega(n_bbo_o(lambda_i_um), omega_i)
    q_i = -q_s
    theta_i_bbo = np.arcsin(q_i / k_i_bbo)

    F1 = (
        L1
        * sinc(dk_bbo * L1 / 2)
        * np.exp(1j * dk_bbo * L1 / 2)
    )

    gap_phase = np.exp(
        1j * (
            dk_bbo * L1
            + dk_air * l_air
            + dk_caco3 * l_caco3
        )
    )

    F2 = (
        L2
        * sinc(dk_bbo * L2 / 2)
        * np.exp(1j * dk_bbo * L2 / 2)
        * gap_phase
    )

    F = F1 + F2

    return F


def angle_distr(x, d_mm, alpha, beta, L, K):
    L1 = 3.0e-3         # BBO 1, m
    L2 = 4.3e-3         # BBO 2, m
    l_air = 16.0e-2        # air gap, m
    # d_mm = 1.800245
    d = d_mm * 1e-3
    l_caco3 = d

    # Накачка
    lambda_p_nm = 404.0
    lambda_p = lambda_p_nm * 1e-9
    w_p = 2 * np.pi * c / lambda_p

    # Подбор угла синхронизма, замыкающегося на 808 нм
    psi_cut = find_bbo_cut_angle_for_degenerate_collinear

    # Поляризация в кальците
    calcite_pol = {
        "p": "o",
        "s": "e",
        "i": "e",
    }
    # L = -0.009425
    # K = 0.0000455
    theta = L + K * x  # связь пикселя с углом параметрики

    # Рассчетные сетки
    # l_calc -- global var!!!
    lambda_s_nm_grid = np.array([l_calc, ])
    omega_s_grid = 2 * np.pi * c / (lambda_s_nm_grid * 1e-9)
    theta_s_grid = theta

    omega_s, theta_s = np.meshgrid(omega_s_grid, theta_s_grid)

    F = JSA(
        omega_s=omega_s,
        theta_s=theta_s,
        L1=L1,
        L2=L2,
        l_air=l_air,
        l_caco3=d,
        calcite_pol=calcite_pol
    )
    S_w_theta = np.abs(F)**2
    J_nm = 2 * np.pi * c * 1e9 / (lambda_s_nm_grid ** 2)
    S_lambda_theta = S_w_theta * J_nm[np.newaxis, :]
    S = S_lambda_theta / np.max(S_lambda_theta)
    S = alpha * S + beta

    return np.asarray(S, dtype=float).ravel()

In [5]:
# Неизменяемые параметры

c = 299_792_458  # [m/s]

l_p = 404.0 * 1e-9
w_p = 2 * np.pi * c / l_p

psi_cut = find_bbo_cut_angle_for_degenerate_collinear()
psi_cut_deg = np.rad2deg(psi_cut)

print(psi_cut, "°")

0.5047770009018182 °


In [6]:
L1 = 3e-3
L2 = 3e-3
l_caco3 = 0
calcite_pol = {
    'p': 'o',
    's': 'e',
    'i': 'e'
}

l_air

In [ ]:
omega_s = 